<a href="https://colab.research.google.com/github/keshavbaviskar33-sudo/AI-Credit-Analyst-Copilot/blob/main/01_initial_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("utkarshx27/american-companies-bankruptcy-prediction-dataset")

print("Path to dataset files:", path)

100%|██████████| 4.47M/4.47M [00:00<00:00, 131MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/utkarshx27/american-companies-bankruptcy-prediction-dataset/versions/3


In [5]:
import pandas as pd
import numpy as np
import os # Import the os module to list directory contents

# List the contents of the directory to verify file names
print(f"Contents of {path}: {os.listdir(path)}")

# The original code was trying to read 'data.csv', but a FileNotFoundError occurred.
# Based on the Kaggle dataset, 'bankruptcy.csv' is a more likely primary data file.
# Correcting the filename based on os.listdir output.
df = pd.read_csv(os.path.join(path, "american_bankruptcy.csv"))
df.head()

Contents of /root/.cache/kagglehub/datasets/utkarshx27/american-companies-bankruptcy-prediction-dataset/versions/3: ['american_bankruptcy.csv']


,company_name,status_label,year,X1,X2,X3,X4,X5,X6,X7,...,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18
0,C_1,alive,1999,511.267,833.107,18.373,89.031,336.018,35.163,128.348,...,1024.333,740.998,180.447,70.658,191.226,163.816,201.026,1024.333,401.483,935.302
1,C_1,alive,2000,485.856,713.811,18.577,64.367,320.590,18.531,115.187,...,874.255,701.854,179.987,45.790,160.444,125.392,204.065,874.255,361.642,809.888
2,C_1,alive,2001,436.656,526.477,22.496,27.207,286.588,-58.939,77.528,...,638.721,710.199,217.699,4.711,112.244,150.464,139.603,638.721,399.964,611.514
3,C_1,alive,2002,396.412,496.747,27.172,30.745,259.954,-12.410,66.322,...,606.337,686.621,164.658,3.573,109.590,203.575,124.106,606.337,391.633,575.592
4,C_1,alive,2003,432.204,523.302,26.680,47.491,247.245,3.504,104.661,...,651.958,709.292,248.666,20.811,128.656,131.261,131.884,651.958,407.608,604.467


In [7]:
df.shape

(78682, 21)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78682 entries, 0 to 78681
Data columns (total 21 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   company_name  78682 non-null  object 
 1   status_label  78682 non-null  object 
 2   year          78682 non-null  int64  
 3   X1            78682 non-null  float64
 4   X2            78682 non-null  float64
 5   X3            78682 non-null  float64
 6   X4            78682 non-null  float64
 7   X5            78682 non-null  float64
 8   X6            78682 non-null  float64
 9   X7            78682 non-null  float64
 10  X8            78682 non-null  float64
 11  X9            78682 non-null  float64
 12  X10           78682 non-null  float64
 13  X11           78682 non-null  float64
 14  X12           78682 non-null  float64
 15  X13           78682 non-null  float64
 16  X14           78682 non-null  float64
 17  X15           78682 non-null  float64
 18  X16           78682 non-nu

In [11]:
df.isnull().sum()

,0
company_name,0
status_label,0
year,0
X1,0
X2,0
X3,0
X4,0
X5,0
X6,0
X7,0


In [13]:
df['status_label'].value_counts(normalize=True)

,proportion
status_label,
alive,0.933657
failed,0.066343


In [14]:
df['company_name'].nunique()

8971

Interest coverage not computed — no interest expense field in source data


In [15]:
df['current_ratio'] = df['X1'] / df['X14']
df['debt_to_equity'] = df['X11']/(df['X10']-df['X17'])
df['net_profit_margin']= df['X6']/df['X16']
df[['current_ratio','debt_to_equity','net_profit_margin']].head()

,current_ratio,debt_to_equity,net_profit_margin
0,3.120983,0.531485,0.034328
1,3.874697,0.529044,0.021196
2,2.902063,0.701723,-0.092277
3,1.947253,0.558185,-0.020467
4,3.292707,0.824260,0.005375


In [16]:
df['current_ratio'].isna().sum()

np.int64(0)

In [17]:
df['debt_to_equity'].isna().sum()

np.int64(0)

In [18]:
df['net_profit_margin'].isna().sum()

np.int64(0)

In [19]:
np.isinf(df[['current_ratio','debt_to_equity','net_profit_margin']]).sum()

,0
current_ratio,0
debt_to_equity,2
net_profit_margin,0


In [20]:
df[np.isinf(df['debt_to_equity'])][['company_name','year','status_label']]

,company_name,year,status_label
64769,C_6980,2002,failed
69826,C_7561,2007,alive


In [21]:
df['debt_to_equity'][~np.isinf(df['debt_to_equity'])].max()

75264.24999964063

In [22]:
df['debt_to_equity'].sort_values(ascending=False).head(15)

,debt_to_equity
64769,inf
69826,inf
61872,7.526425e+04
22895,1.338000e+04
61093,1.188877e+04
59863,4.462850e+03
22786,3.526545e+03
10750,3.096268e+03
70097,2.640889e+03
19452,2.170615e+03


Changing the inf values of debt to equity ratio to 99%ile winsorized value


In [23]:
cap = df['debt_to_equity'][~np.isinf(df['debt_to_equity'])].quantile(0.99)
df['debt_to_equity'] = df['debt_to_equity'].replace([np.inf, -np.inf], cap)

In [24]:
print(cap)

10.25873829219188


In [26]:
np.isinf(df[['current_ratio','debt_to_equity','net_profit_margin']]).sum()

,0
current_ratio,0
debt_to_equity,0
net_profit_margin,0
